In [ ]:
### Install requirements
from pathlib import Path

REPOSITORY = "https://github.com/lmingari/olot-course.git"
PROJECT_FILE = "pyproject.toml"

if not Path(PROJECT_FILE).exists():
    !git clone -q {REPOSITORY}
    %cd olot-course
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu
    !pip install -q -e ".[maps,data,ml]"

# 2.2 A multilayer perceptron (MLP) for classification

---

In this practice, we will use a simple **multilayer perceptron (MLP)** to classify the impact of tephra fallout at different locations on La Palma.

* We use the volcanic tephra deposition dataset from the 2021 Tajogaite Eruption on La Palma reported by [Shatto et al. (2024)][dataset]

* The dataset contains **415 in-situ field measurements** and **66 additional values** estimated from surface changes in areas unsuitable for in-situ sampling.

* The dataset is available on [Zenodo][zenodo]

| La Palma - Tephra depth measurements |
| --- |
| <img src="https://ars.els-cdn.com/content/image/1-s2.0-S2352340923009800-gr2.jpg" width=450/> |
| **Figure 1:** Tephra depth measurements from [Shatto et al. (2024)][dataset] |

[dataset]: https://doi.org/10.1016/j.dib.2023.109949
[zenodo]: https://doi.org/10.5281/zenodo.8338991
[paper-fig]: https://ars.els-cdn.com/content/image/1-s2.0-S2352340923009800-gr2.jpg

## Definition of impact classes

We define three classes of tephra fallout impact based on deposit thickness:

|Impact    | Thickness range   | Class |
|----------|:-----------------:|:-----:|
| Low      | < 1 mm            | 0     |
| Moderate | 1 - 100 mm        | 1     |
| High     | > 100 mm          | 2     |

> &#10004; **Goal:** \
>  Train an MLP to predict the impact class at a given location from its (lat, lon)

## Classification and loss function

In the previous practice, the neural network was used for **regression**: the model
predicted a continuous value.

Here, we want to predict one of a small number of **classes**. This is a
**classification** problem.

The model produces one output value, called a **logit**, for each class.

Internally, these logits are converted into values that can be interpreted as class
probabilities and compared with the observed class via the **cross-entropy loss
function**.

The class with the largest logit is the model's prediction:

$$
\hat{y} = \underset{k}{\operatorname{argmax}}\; z_k.
$$

| Multilayer perceptron for classification |
| --- |
| <img src="figs/mlp-classifier.svg" width=600/> |

## &#128221; Example 2.4: Defining datasets

### Loading data
Let's load the tephra deposit measurements from `data/palma-deposit.csv`:

In [ ]:
import pandas as pd
df = pd.read_csv('data/palma-deposit.csv')
df

### Defining target categories

The dataset contains the deposit thickness in centimetres. We convert it into the three impact classes defined above:

In [ ]:
## Define the categories

df['category'] = 0
df.loc[df['thickness_cm']>0.1,'category'] = 1
df.loc[df['thickness_cm']>10,'category']  = 2
df

The resulting `category` column is our target variable.

### Check dataset balance

Let's check how many observations belong to each impact class:

In [ ]:
## Check dataset balance
_ = df['category'].value_counts().plot(kind='bar')

The classes are not equally represented. This is called **class imbalance** and can affect how well a classifier learns the different classes.

A classifier trained on an imbalanced dataset may favour the most frequent class!

### Data visualisation

In [ ]:
from modules.plotting import MapPlotter

plotter = MapPlotter()

conf_plot = {
    0: dict(color = "tab:green",  marker = '+', label = "L"),
    1: dict(color = "tab:orange", marker = 'x', label = "M"),
    2: dict(color = "tab:red",    marker = '.', label = "H"),
}

for category, group in df.groupby("category"):
    plotter.add_marker(
        group["lon"],
        group["lat"],
        alpha = 0.4,
        **conf_plot[category],
    )
plotter.add_legend()

### Define features and targets

In [ ]:
features = ['lat','lon']
X = df[features].values
y = df['category'].values

### Split dataset

We split the observations into a training set and a validation set:

In [ ]:
from modules.dataset import train_validation_split

X_train, X_val, y_train, y_val = train_validation_split(X, y, train_fraction=0.8)

### Standardise input data

The two input features, latitude and longitude, have different numerical ranges.
We therefore standardise each feature to have approximately zero mean and unit variance:

$$
x \to \frac{x-\mu}{\sigma}
$$

Importantly, $\mu$ and $\sigma$ are calculated **using the training data only** and then applied to the validation data.

In [ ]:
from modules.dataset import Standardize, SupervisedDataset

# Compute normalization stats from training data only
mean = X_train.mean(axis=0)
std  = X_train.std(axis=0)

# Define transform
transform = Standardize(mean,std)

# Datasets
train_dataset = SupervisedDataset(X_train, y_train, transform=transform)
val_dataset   = SupervisedDataset(X_val, y_val, transform=transform)

## &#128221; Example 2.5: training the classifier

We use the `Trainer` defined in the previous practice. For classification, we only need to define how the loss is calculated for each batch.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

In [ ]:
from modules.training import Trainer

###
### Define trainer class
###
class ClassificationTrainer(Trainer):
    def get_batch_results(self, batch):
        X, y = batch
        prediction = self.model(X)
        loss = self.criterion(prediction, y)

        accuracy = (
            prediction.argmax(dim=1) == y
        ).float().mean()

        return {
            "loss": loss,
            "accuracy": accuracy,
        }

In [ ]:
###
### Define a simple classifier
###
class MultiLayerPerceptron(nn.Module):
    def __init__(self, in_dim=2, hidden=64, out_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim)  # output logits
        )

    def forward(self, x):
        return self.net(x)

The same training workflow can be used here:

In [ ]:
### Training configuration
conf = {
    'BATCH_SIZE': 16,
    'LEARNING_RATE': 1E-3,
    'NUM_EPOCHS': 800,
}

In [ ]:
model = MultiLayerPerceptron()

trainer = ClassificationTrainer(
    model     = model,
    criterion = nn.CrossEntropyLoss(),
    optimizer = torch.optim.Adam(model.parameters(), lr=conf['LEARNING_RATE']),
)

In [ ]:
# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=conf['BATCH_SIZE'], shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=conf['BATCH_SIZE'], shuffle=False)

history = trainer.fit(train_loader, val_loader, epochs=conf['NUM_EPOCHS'])

In [ ]:
history

In [ ]:
history.plot(
    x="epoch",
    y=["train_loss", "val_loss"],
)

In [ ]:
history.plot(
    x="epoch",
    y=["train_accuracy", "val_accuracy"],
)

## &#128221; Example 2.6: decision regions

The trained MLP can be used to predict the impact class at any location.

We evaluate the model on a regular grid of latitude and longitude values and visualise
the resulting **decision regions**.

In [ ]:
from modules.helper import get_decision_regions
from modules.plotting import DataPlotter

data = get_decision_regions(model, transform)
plotter = DataPlotter(data)

plotter.pcolormesh()

conf_plot = {
    0: dict(color = "tab:green"),
    1: dict(color = "tab:orange"),
    2: dict(color = "tab:red"),
}

for category, group in df.groupby("category"):
    plotter.add_marker(
        group["lon"],
        group["lat"],
        alpha = 0.4,
        marker = '.',
        s = 5,
        **conf_plot[category],
    )

## &#9998; Exercise 1: Effect of the activation function

* Redefine the model class `MultiLayerPerceptron` by removing the `ReLU` activation functions
* Retrain the classifier
* How are the decision regions modified?

## &#9998; Exercise 2: Overfitting

Modify the training setup to increase overfitting:

* Decrease the learning rate
* Increase the number of epochs
* Add more neurons or more layers

Observe how the decision regions and validation loss change.